In [4]:
pip install instanseg-torch

  Using cached instanseg_torch-0.1.1-py3-none-any.whl.metadata (12 kB)
  Using cached einops-0.8.2-py3-none-any.whl.metadata (13 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
Using cached instanseg_torch-0.1.1-py3-none-any.whl (126 kB)
Using cached einops-0.8.2-py3-none-any.whl (65 kB)
   ---------------------------------------- 0.0/122.0 MB ? eta -:--:--
    --------------------------------------- 1.6/122.0 MB 7.7 MB/s eta 0:00:16
    --------------------------------------- 2.9/122.0 MB 7.0 MB/s eta 0:00:18
   - -------------------------------------- 3.4/122.0 MB 5.9 MB/s eta 0:00:21
   - -------------------------------------- 4.2/122.0 MB 5.1 MB/s eta 0:00:23
   - -------------------------------------- 5.5/122.0 MB 5.2 MB/s eta 0:00:23
   -- ------------------------------------- 6.6/122.0 MB 5.1 MB/s eta 0:00:23
   -- ------------------------------------- 7.6/122.0 MB 5.2 MB/s eta 0:00:23
   -- ------------------------------------- 8.9/122.0 MB 5.2 MB/s eta 0:00:2

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cellpose 2.2.3 requires imagecodecs, which is not installed.
cellpose 2.2.3 requires llvmlite, which is not installed.
cellpose 2.2.3 requires numba>=0.53.0, which is not installed.


In [ ]:
pip uninstall Instanseg

In [5]:
import numpy as np
import pandas as pd
import skimage as ski
import matplotlib.pyplot as plt
import tifffile
import zarr
import os
import xml.etree.ElementTree as ET
import napari
from importlib import reload
from instanseg import InstanSeg
import load_assess_image as load_assess
import qc_widget

from concurrent.futures import ThreadPoolExecutor   # was ProcessPoolExecutor


ImportError: cannot import name 'InstanSeg' from 'instanseg' (d:\Thu\conda_envs\instanseg\Lib\site-packages\instanseg\__init__.py)

In [ ]:
from instanseg import InstanSeg
from skimage.measure import regionprops_table
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))  # shows your GPU name

True
NVIDIA GeForce RTX 3060


In [3]:
reload(load_assess)

<module 'load_assess_image' from 'r:\\Thu\\From_2026\\6. Other\\DL_Summer_2026\\notebook\\load_assess_image.py'>

In [ ]:
root_dir=rf'D:\thu\HTAN\images'

In [14]:
# view_list=['CD45','CD44','PDPN','aSMA','panCK','HLA-DPB1'] either PDPN or Vim
view_list_2=['CD45','aSMA','panCK','Vim','PDPN']

In [ ]:
save_dir=rf"{root_dir}\mask\NST"
os.makedirs(save_dir,exist_ok=True)

In [16]:
main_dir = rf'D:\thu\HTAN\images\NST'
file_name= 'LSP12021-C10.ome.tif'
image_test = load_assess.AssessImage(main_dir, file_name)

In [22]:
def process_MIP(args):
    main_dir, filename = args
    image = load_assess.AssessImage(main_dir, filename)

    image_full = zarr.open(image.loader.image_store, mode='r')
    dna_idx = 0
    dna = np.array(image_full['0'][dna_idx])
    cyto = image.max_projection(view_list_2, resolution_level=0)

    image.loader.tif.close()
    image.loader.image_store.close()
    return filename, dna, cyto


In [27]:
list_file = []
for i in os.listdir(main_dir):
    if i.endswith('.ome.tif'):
        image_test = load_assess.AssessImage(main_dir, i)
        if 'pRB' in image_test.biomarker_channel:
            list_file.append(i)

In [28]:

# --- parallel MIP (CPU/IO bound) ---
with ThreadPoolExecutor(max_workers=26) as pool:
    mip_results = list(pool.map(process_MIP, [(main_dir, i) for i in list_file]))

# --- sequential segmentation (GPU bound) ---
model = InstanSeg('fluorescence_nuclei_and_cells', device='cuda')
seg_results = {}

for filename, dna, cyto in mip_results:
    stack = np.stack([dna, cyto])
    labeled_output = model.eval_medium_image(stack, pixel_size=0.65)
    
    # Save mask in NPZ form - faster to reload
    stem = filename.replace('.ome.tif', '')   # → 'TNP-TMA-29_A7-ILC-3A'
    output_path = os.path.join(save_dir, stem + '.npz')
    nuclei_mask = labeled_output[0][0, 0].cpu().numpy().astype(int)
    cells_mask  = labeled_output[0][0, 1].cpu().numpy().astype(int)
    np.savez_compressed(output_path, nuclei=nuclei_mask, cells=cells_mask)
















Model fluorescence_nuclei_and_cells version 0.1.1 already downloaded in d:\conda_envs\instanseg\Lib\site-packages\instanseg\utils\../bioimageio_models/, loading


 24%|██▍       | 231/961 [00:18<01:22,  8.89it/s]

Reached maximum number of iterations - this is not expected!


 34%|███▍      | 329/961 [00:22<01:04,  9.77it/s]

Reached maximum number of iterations - this is not expected!


...

Reached maximum number of iterations - this is not expected!


 81%|████████  | 601/744 [00:51<00:13, 10.44it/s]

Reached maximum number of iterations - this is not expected!
Reached maximum number of iterations - this is not expected!


 58%|█████▊    | 332/576 [00:33<00:27,  8.96it/s]

Reached maximum number of iterations - this is not expected!


In [29]:
len(list_file)

469

In [19]:
save_dir_mip=rf"{main_dir}/MIP_cyto"
os.makedirs(save_dir_mip, exist_ok=True)

for i in range(len(mip_results)):
    cyto_mip=mip_results[i][2]
    stem = mip_results[i][0].replace('.ome.tif', '')   # → 'TNP-TMA-29_A7-ILC-3A'
    output_path = os.path.join(save_dir_mip, stem + '.npz')
    np.savez_compressed(output_path, cyto_mip=cyto_mip)

# TENSOR
* labeled_output is a tuple. InstanSeg returns (segmentation_tensor, ...). So labeled_output[0] gets the segmentation tensor.
* The tensor shape is (batch, channels, height, width):
labeled_output[0].shape → (1, 2, H, W)
                           │  │
                           │  └── 2 channels: 0=nuclei, 1=whole cell
                           └───── batch size = 1 (one image)
* labeled_output[0][0, 0]   # batch 0, channel 0 → nuclei mask  (H, W)
* labeled_output[0][0, 1]   # batch 0, channel 1 → cells mask   (H, W)

* .cpu()       # move tensor from GPU memory → CPU memory (required before numpy)
* .numpy()     # convert PyTorch tensor → numpy array
*.astype(int) # each pixel value = instance ID (0=background, *1=cell1, 2=cell2...)


In [ ]:
list_file

In [64]:
filename=list_file_with_panCK[45] # 5,6 is bad
stem = filename.replace('.ome.tif', '')   # → 'TNP-TMA-29_A7-ILC-3A'
output_path_mip = os.path.join(save_dir_mip, stem + '.npz')
data_mip = np.load(output_path_mip)
mip = data_mip['cyto_mip']

output_path = os.path.join(save_dir, stem + '.npz')
data = np.load(output_path)
nuclei_mask = data['nuclei']
cells_mask  = data['cells']


image_test=load_assess.AssessImage(main_dir,filename)
viewer = image_test.view_images(biomarkers_of_interest=['dna'], resolution_level=0)
viewer.add_image(mip, name='MIP',blending='additive', colormap='magenta')
viewer.add_labels(nuclei_mask, name='nuclei')
viewer.add_labels(cells_mask,  name='cells')

Computing stats: 100%|██████████| 40/40 [00:00<00:00, 113.25it/s]


<Labels layer 'cells' at 0x1fa4553b2d0>

In [ ]:
model = InstanSeg('fluorescence_nuclei_and_cells',device='cuda')


Model fluorescence_nuclei_and_cells version 0.1.1 already downloaded in d:\conda_envs\instanseg\Lib\site-packages\instanseg\utils\../bioimageio_models/, loading


In [ ]:
stack=np.stack(arrays=(image_dna,cyto))
stack.shape

(2, 2672, 2672)

In [86]:
for element in image_test.loader.root.iter():
    if element.tag.endswith('Pixels'):
        print(element.attrib)

{'ID': 'Pixels:0', 'DimensionOrder': 'XYCZT', 'Type': 'uint16', 'SizeX': '2672', 'SizeY': '2672', 'SizeC': '40', 'SizeZ': '1', 'SizeT': '1', 'PhysicalSizeX': '0.6499999761581421', 'PhysicalSizeXUnit': 'µm', 'PhysicalSizeY': '0.6499999761581421', 'PhysicalSizeYUnit': 'µm'}


In [87]:
model = InstanSeg('fluorescence_nuclei_and_cells', device='cuda')
labeled_output = model.eval_small_image(stack, pixel_size=0.65)


Model fluorescence_nuclei_and_cells version 0.1.1 already downloaded in d:\conda_envs\instanseg\Lib\site-packages\instanseg\utils\../bioimageio_models/, loading


In [88]:
print(type(labeled_output))
print(len(labeled_output))
for i, item in enumerate(labeled_output):
    print(i, type(item), item.shape if hasattr(item, 'shape') else item)


<class 'tuple'>
2
0 <class 'torch.Tensor'> torch.Size([1, 2, 2672, 2672])
1 <class 'torch.Tensor'> torch.Size([1, 2, 2672, 2672])


In [89]:
nuclei_mask = labeled_output[0][0, 0].cpu().numpy().astype(int)
cells_mask  = labeled_output[0][0, 1].cpu().numpy().astype(int)

viewer = image_test.view_images(biomarkers_of_interest=['dna'], resolution_level=0)
viewer.add_image(cyto, name='MIP')
viewer.add_labels(nuclei_mask, name='nuclei')
viewer.add_labels(cells_mask,  name='cells')


<Labels layer 'cells' at 0x216f58a5f10>

In [31]:
save_dir

'D:\\thu\\HTAN\\images\\NST\\mask\\NST'

In [ ]:


# shape props once — independent of biomarker
shape_props = regionprops_table(
    cells_mask,
    properties=['label', 'area', 'centroid', 'equivalent_diameter_area',
                'solidity', 'eccentricity', 'axis_major_length', 'axis_minor_length', 'perimeter']
)
df = pd.DataFrame(shape_props)

# intensity per biomarker — rename columns to avoid collision, merge on label
for biomarker, channel_index in image_test.biomarker_channel.items():
    image_biomarker = np.array(open_image_test[str(resolution_level)][int(channel_index)])
    intensity_props = regionprops_table(
        cells_mask,
        intensity_image=image_biomarker,
        properties=['label', 'intensity_mean', 'intensity_median', 'intensity_std']
    )
    intensity_df = pd.DataFrame(intensity_props).rename(columns={
        'intensity_mean':   f'{biomarker}_mean',
        'intensity_median': f'{biomarker}_median',
        'intensity_std':    f'{biomarker}_std',
    })
    df = df.merge(intensity_df, on='label')

output_path = os.path.join(r'r:\Thu\From_2026\6. Other\DL_Summer_2026\output',
                           file_name.replace('.ome.tif', '_cells.csv'))
df.to_csv(output_path, index=False)
print(f"Saved {len(df)} cells → {output_path}")


Saved 11060 cells → r:\Thu\From_2026\6. Other\DL_Summer_2026\output\LSP12021-C10_cells.csv
